In [ ]:
# import a bunch of stuff
import networkx as nx
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
from matplotlib import rc
import time
import random
import math
from collections import Counter

## Question 1

#### Question 1a

In [ ]:
def grow_barabasi_albert_net(N, m, seed=None, debug=False):
    """
    This function creates a scale-free network using barabasi-albert model.

    It loops through and adds the proper amount of nodes (specified as param).
    For each node, it connects via m links to existing nodes. 

    @param N (int): The number of nodes
    @param m (int): Number of edges formed for each new node
    @param seed (int?): The seed for the network (optional)
    @param debug (bool): Whether or not you want print statements

    @return the scale-free network
    @return an array of degree sequences at N^x
    """

    # Create the graph
    degree_sequences = []
    simple_graph = nx.complete_graph(4)
    start_graph = nx.MultiGraph(simple_graph) # multilinks are allowed

    # Set our seed
    random.seed(seed)

    # Our time counter
    # Start with 4 because the network always starts with 4 nodes
    t = 4
    edge_list = list(start_graph.edges) 

    # modularity array 
    modularity = [0,0,0,0]

    while t < N:
        t = t + 1

        # add node
        start_graph.add_node(t)

        edge_list = list(start_graph.edges) 

        # Now add the m edges
        # Note that this doesn't allow self-links
        # But I don't think that's a requirement for BA; according to the book
        # BA leaves many mathematical details open
        for i in range(0,m):
            # get random edge
            chosen_edge = edge_list[random.randint(0,len(edge_list)-1)]
            if debug: print(f"chosen_edge: {chosen_edge}")

            #randomly choose one edge of that link
            chosen_node = chosen_edge[random.randint(0,1)]
            if debug: print(f"Chosen node: {chosen_node}")

            # attach the new edge 
            start_graph.add_edge(t, chosen_node)
            edge_list.append((t, chosen_node))

            # calculate and add the modularity 
            louv = nx.community.louvain_communities(start_graph)
            modularity.append(nx.algorithms.community.modularity(start_graph, louv))

    return start_graph, modularity

In [ ]:
# make the BA networks
BA_networks = []
mods = []
N = 400
for m in range(1,6):
    temp_G, modularity = grow_barabasi_albert_net(N, m)
    BA_networks.append(temp_G)
    mods.append(modularity)

    print(f"Completed network with m {m}")


In [ ]:
fig, ax = plt.subplots(1,1,figsize=(8,4),dpi=100)
markers = 'o', 's', 'D', '^', '<', 

for m in range(0,5):
    vals = {i: m for i,m in enumerate(mods[m]) if i % 50 == 0 and i > 49}
    ax.plot(list(vals.keys()), list(vals.values()), marker=markers[m],lw=1, linestyle='-', label=f"m = {m+1}")

ax.set_xlabel(r"Size", fontsize='medium')
ax.set_ylabel(r"Modularity", fontsize='medium')
ax.grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')
ax.set_title(f'Modularity in Scale-free Networks', fontsize='large')
ax.set_xlim(right=400)
ax.set_xlim(0)

ax.legend()

plt.savefig('images/modularity_graph.png', dpi=425, bbox_inches='tight')

plt.show()

In [ ]:
mat1 = np.array([[1,0,1],[1,0,1],[0,1,0]])
mat1@mat1

#### Question 2b

In [ ]:
import graph_tool.all as gt

In [ ]:
# import polblog data
pol_G = gt.Graph(directed=True)
with open('data/polblogs_nodes_class.tsv','r') as fp:
    for line in fp:
        node_id, node_label, node_political = line.strip().split('\t')
        political_orientation = 'liberal' if node_political == '0' else 'conservative'
        pol_G.add_vertex()

with open('data/polblogs_edges_class.tsv','r') as fp:
    for line in fp:
        source_node_id, target_node_id = line.strip().split('\t')
        pol_G.add_edge(source_node_id, target_node_id)

print("Number of nodes: %d" % pol_G.num_vertices())
print("Number of edges: %d" % pol_G.num_edges())

In [ ]:
pol_G = gt.collection.ns["polblogs"]
g_connected = gt.extract_largest_component(pol_G, directed=False)

In [ ]:
from scipy.io import mmread

pos = gt.sfdp_layout(g_connected)
gt.graph_draw(g_connected, pos, vertex_fill_color=g_connected.vp.value, vcmap=matplotlib.cm.coolwarm,
           vcnorm=matplotlib.colors.Normalize(vmin=0, vmax=1))

##### Now Degree Corrected and Not

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(16,7),dpi=200)

partitioned_no_deg = gt.minimize_blockmodel_dl(g_connected, multilevel_mcmc_args=dict(B_min=2, B_max=2), state_args=dict(deg_corr=False))
a0 = partitioned_no_deg.draw(mplfig=ax[0], pos=g_connected.vp._pos)
a0.fit_view()

partitioned_deg = gt.minimize_blockmodel_dl(g_connected, multilevel_mcmc_args=dict(B_min=2, B_max=2), state_args=dict(deg_corr=True))
a1 = partitioned_deg.draw(mplfig=ax[1], pos=g_connected.vp._pos)
a1.fit_view()

ax[0].set_title('Non-degree Corrected SBM', fontsize='large')
ax[1].set_title('Degree Corrected SBM', fontsize='large')
ax[0].set_axis_off()
ax[1].set_axis_off()

plt.savefig('images/polblogs_sbg.png', dpi=425, bbox_inches='tight')

plt.show()

#### Question 1c

In [ ]:
print(f"The description length non-degree corrected: {partitioned_no_deg.entropy()}")
print(f"The description length degree corrected: {partitioned_deg.entropy()}")

In [ ]:
b=np.random.randint(0, 2, g_connected.num_vertices())

partitioned_no_deg = gt.BlockState(g_connected, b=b.copy(), deg_corr=False)
partitioned_deg = gt.BlockState(g_connected, b=b.copy(), deg_corr=True)

no_deg_entr_list = []
deg_entr_list = []

for i in range(1000):
    partitioned_no_deg.mcmc_sweep(niter=1)
    no_deg_entr_list.append(partitioned_no_deg.entropy())
    partitioned_deg.mcmc_sweep(niter=1)
    deg_entr_list.append(partitioned_deg.entropy())

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(8,4),dpi=100)
markers = ['o', 's']

ax.plot(no_deg_entr_list, marker='o',lw=1, linestyle='-', label=f"No Degree Correction", markersize=2)
ax.plot(deg_entr_list, marker='o',lw=1, linestyle='-', label=f"Degree Correction", markersize=2)


ax.set_xlabel(r"Iteration", fontsize='medium')
ax.set_ylabel(r"Entropy (description length)", fontsize='medium')
ax.grid(linewidth=1.5, color='#999999', alpha=0.2, linestyle='-')
ax.set_title(f'Description Length over Iterations', fontsize='large')
ax.legend()

plt.savefig('images/description_length.png', dpi=425, bbox_inches='tight')
plt.show()

## Question 2

#### Question 2a

In [ ]:
G = nx.karate_club_graph()

In [ ]:
def get_commute_time(G, i, j, max_steps=10000):
    """
        Function that measures the commute time for a random walker
        to get from node i to node j and then back to node i. 

        @param (int) i: the starting node
        @param (int) j: the destination node

        @return: The path, a list of nodes the random walker hit
    """
    current_node = i

    is_return = False

    steps = 0

    while steps < max_steps:
        neighbors = list(G.neighbors(current_node))
        if neighbors:
            next_node = np.random.choice(neighbors)
            current_node = next_node
        else:
            print("Failure. This node has no neighbors.")
            break
        if is_return == False and current_node == j:
            is_return = True
        if is_return and current_node == i:
            break
        steps += 1
    
    return steps

In [ ]:
for i in range(3):
    i = np.random.choice(G.nodes())
    j = i
    while i == j:
        j = np.random.choice(G.nodes())
    print(f"Starting random walk from node {i} to node {j}...")
    path = get_commute_time(G, i,j)
    print(f"The commute time was {len(path)}.\n")

#### Question 2b

In [ ]:
N = 500
p = 8/(N-1)
m = 4

ER_G = nx.erdos_renyi_graph(n=N, p=p)
ER_G_cc = ER_G.subgraph(max(nx.connected_components(ER_G), key=len)).copy()

BA_G = nx.barabasi_albert_graph(N, m)
BA_G_cc = BA_G.subgraph(max(nx.connected_components(BA_G), key=len)).copy()

In [ ]:

plt.figure(figsize=(4, 4))
pos = nx.spring_layout(ER_G_cc)
nx.draw(ER_G_cc, with_labels=False, pos=pos, node_color='lightblue', edge_color='gray', node_size=20)
plt.title("ER Connected Graph")
plt.show()

In [ ]:
N = 500
p = 8/(N-1)
m = 4

ER_avg = []
BA_avg = []

for i in range(10):
    ER_G = nx.erdos_renyi_graph(n=N, p=p)
    ER_G_cc = ER_G.subgraph(max(nx.connected_components(ER_G), key=len)).copy()

    BA_G = nx.barabasi_albert_graph(N, m)
    BA_G_cc = BA_G.subgraph(max(nx.connected_components(BA_G), key=len)).copy()

    # Average Commute time for ER network
    er_commute_times = []
    er_nodes = list(ER_G_cc.nodes())

    for idx,i in enumerate(er_nodes):
        for j in er_nodes[idx+1:]:
            er_commute_times.append(get_commute_time(ER_G_cc, i,j))
        if idx % 50 == 0:
            print(f"ER: Completed node {i} and index {idx}.")

    avg_commute_time = np.mean(er_commute_times)
    ER_avg.append(avg_commute_time)

    # Average Commute time for BA Network 
    ba_commute_times = []
    ba_nodes = list(BA_G_cc.nodes())

    for idx,i in enumerate(ba_nodes):
        for j in ba_nodes[idx+1:]:
            ba_commute_times.append(get_commute_time(BA_G_cc, i,j))
        if idx % 50 == 0:
            print(f"BA: Completed node {i} and index {idx}.")

    ba_avg_commute_time = np.mean(ba_commute_times)
    BA_avg.append(ba_avg_commute_time)

    print("Completed iteration {i}.")

er_avg = np.mean(ER_avg)
ba_avg = np.mean(BA_avg)

In [ ]:
print(f"ER average: {er_avg}")
print(f"BA average: {ba_avg}")

In [ ]:

print(np.median(ba_commute_times))
print(np.std(ba_commute_times))
print(np.median(er_commute_times))
print(np.std(er_commute_times))

## Question 3

Eigenvector Centrality

#### Question 3b

In [ ]:

import textwrap
from matplotlib.colors import ListedColormap
import matplotlib.patches as patches

fig = plt.figure(figsize=(14, 20), layout="constrained", dpi=150)

subfigs = fig.subfigures(nrows=3, ncols=1, height_ratios=[1,1.2,6])

#############
# Title
#############
ax_top = subfigs[0].subplots(1) 

box_style = dict(boxstyle="square,pad=0.8", facecolor="#6D8CBB", alpha=0.8,  edgecolor='none')
ax_top.text(
    .5, .5,                       
    "Eigenvector Centrality", 
    transform=ax_top.transAxes,
    fontsize=40,
    color="white",
    ha='center', 
    va='center',
    bbox=box_style,
    zorder=0,
    fontweight="bold"
)

# rect = patches.Rectangle(
#     (.25, .15), 
#     .55, 
#     .52, 
#     linewidth=2, 
#     edgecolor='white', 
#     #fill=False,
#     zorder=1,
#     transform=ax_top.transAxes,
# )
# ax_top.add_patch(rect)
ax_top.axis('off')

#############
# Description
#############

ax_mid = subfigs[1].subplots(1) 

e_text = "Eigenvector centrality is a method of calculating centrality in which a node is considered more important, or central, if it is connected to other very important nodes. In other words, a node's importance is determined by the sum of it's neighbors' importance, which are determined by the sum of their neighbors' importance... and so forth."
e_text2 = "So.... we need a vector representing each node's value as a sum of it's neighbors' values- for every single node."

wrapped_text = textwrap.fill(e_text, width=80)
ax_mid.text(
    .5,.6,                       
    wrapped_text, 
    transform=ax_mid.transAxes,
    fontsize=16,
    color="#666F7A",
    ha='center', 
)

wrapped_text2 = textwrap.fill(e_text2, width=60)
ax_mid.text(
    .5,.3,                       
    wrapped_text2, 
    transform=ax_mid.transAxes,
    fontsize=16,
    color="#666F7A",
    ha='center', 
    fontweight="bold"
)

########################
#Visualization
#######################
ax_bottom = subfigs[2].subplots(3,2) 
# turn off all axis
for i in range(3):
    for j in range(2):
        ax_bottom[i,j].axis('off')

##### The Network ########
net_ax = ax_bottom[0,0]

# create the network 
G = nx.Graph()
G.add_nodes_from([1, 2, 3])
G.add_edges_from([(1,2), (2,3)])
pos = nx.spring_layout(G)
nx.draw_networkx(
    G,
    pos=pos,
    ax=net_ax,
    with_labels=True,
    node_color="#6D8CBB",
    node_size=800,
    font_weight="bold",
)
net_ax.margins(0.80)
net_ax.set_title(r"Let's look at this simple network:")
# Border
for spine in net_ax.spines.values():
  spine.set_visible(True)
  spine.set_edgecolor("black")  # Border color
  spine.set_linewidth(1)  # Border thickness


##### The Matrix ########
mat_ax = ax_bottom[0,1]

matrix = nx.to_numpy_array(G)

#create matrix annotations
for i in range(matrix.shape[0]):
    mat_ax.text(-1, i, str(i+1), color='black')
    for j in range(matrix.shape[1]):
        mat_ax.text(j, i, str(int(matrix[i, j])), va='center', ha='center', color='black')

cmap = ListedColormap(["white", "lightblue"])
cax = mat_ax.matshow(matrix, cmap=cmap)

mat_ax.margins(0.80)
mat_ax.set_title(r"And it's corresponding adjacency matrix.")

ax_mid.axis('off')
ax[2].axis('off')
plt.show()